In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
    
    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [9]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client
)

In [10]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want to receive a certificate, make sure you submit your project while submissions are still open.'

In [11]:
vs_index.close()